# **2. Limpieza y Construcción de Variables**

## **2.1 Carga del dataset limpio (processed) y verificación**

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

def find_project_root(start: Path, marker: str = "data", max_up: int = 6) -> Path:
    p = start.resolve()
    for _ in range(max_up):
        if (p / marker).exists():
            return p
        p = p.parent
    return start.resolve()

ROOT = find_project_root(Path.cwd())
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

clean_path = DATA_PROCESSED / "Ventas_1_clean.csv"
assert clean_path.exists(), f"No encuentro el archivo limpio en: {clean_path}"

df = pd.read_csv(clean_path, parse_dates=["FECHA"])
print("Ruta:", clean_path)
print("Shape:", df.shape)
df.head(5)

Ruta: C:\Users\juana\olimpica_book\data\processed\Ventas_1_clean.csv
Shape: (345, 16)


,NroReg,FECHA,CENTRO,Estrato,OFERTA_ID,FACTURA,GRUPO_CATEG,PLU_SAP,CANTIDAD,VENTA,DESCUENTO,GRUCOM,VENTA_NETA,OFERTA_ID_NUM,PROMO_FLAG,PRECIO_UNITARIO_NETO
0,45,2026-01-01,1,4,0,2,1,3,2.061,36900,0.000,1,"36,900.000",0,0,"17,903.930"
1,61,2026-01-01,1,4,0,4,1,5,7.095,47900,0.000,1,"47,900.000",0,0,"6,751.233"
2,63,2026-01-01,1,4,0,6,1,7,0.984,30450,0.000,1,"30,450.000",0,0,"30,945.122"
3,83,2026-01-01,1,4,0,8,1,9,0.771,17750,0.000,1,"17,750.000",0,0,"23,022.049"
4,88,2026-01-01,1,4,0,10,1,11,0.402,12150,0.000,1,"12,150.000",0,0,"30,223.881"



### Validación de persistencia y trazabilidad

- Ruta confirmada: `data/processed/Ventas_1_clean.csv`
- Shape cargado: 345 filas y 16 columnas

El incremento de 12 a 16 columnas confirma la incorporación exitosa de variables derivadas estratégicas sin alterar el número de registros originales.

### Estructura actual del dataset

Además de las variables transaccionales originales, ahora se dispone de:

- `VENTA_NETA`: ingreso real por línea.
- `OFERTA_ID_NUM`: versión numérica para análisis.
- `PROMO_FLAG`: indicador binario de promoción.
- `PRECIO_UNITARIO_NETO`: señal económica clave para análisis de elasticidad.

La columna `FECHA` se carga correctamente como tipo fecha (`parse_dates=["FECHA"]`), habilitando análisis temporales y construcción de series de tiempo.

### Lectura de la muestra 

En los primeros registros se observa:

- Coherencia entre `VENTA` y `VENTA_NETA` cuando no hay descuento.
- `PROMO_FLAG = 0` consistente con ausencia de descuento.
- Cantidades fraccionadas que confirman venta por peso en algunos productos.
- Precios unitarios netos consistentes con la relación venta/cantidad.



## **2.2 Normalización de columnas y control de tipos (re-chequeo)**

In [3]:
# (Opcional) Normalizar nombres de columnas a un estándar
# Aquí mantenemos nombres originales, pero validamos tipos críticos.

expected_types = {
    "FECHA": "datetime64[ns]",
    "VENTA": "float",
    "DESCUENTO": "float",
    "VENTA_NETA": "float",
    "CANTIDAD": "float",
}

type_report = {}
for col, t in expected_types.items():
    if col in df.columns:
        type_report[col] = str(df[col].dtype)

type_report

{'FECHA': 'datetime64[us]',
 'VENTA': 'int64',
 'DESCUENTO': 'float64',
 'VENTA_NETA': 'float64',
 'CANTIDAD': 'float64'}


## Validación de tipos de datos en variables críticas

El bloque ejecutado verifica que las variables fundamentales para análisis cuantitativo y modelado tengan el tipo de dato adecuado tras el proceso de limpieza.

### Resultado del reporte de tipos

- `FECHA`: datetime64[us]  
- `VENTA`: int64  
- `DESCUENTO`: float64  
- `VENTA_NETA`: float64  
- `CANTIDAD`: float64  

### Interpretación técnica

**FECHA**  
Se encuentra correctamente tipificada como fecha. Aunque aparece como `datetime64[us]` en lugar de `datetime64[ns]`, funcionalmente es equivalente para análisis temporal en pandas. Esto permite:

- agregaciones por día, semana o mes  
- construcción de variables temporales  
- modelado de series de tiempo  

**VENTA**  
Se encuentra como `int64`. Desde una perspectiva analítica, esto es válido dado que representa valores monetarios enteros.  
Si se requiere homogeneidad con `VENTA_NETA`, podría convertirse a `float64`, pero no es estrictamente necesario.

**DESCUENTO y VENTA_NETA**  
Correctamente tipificadas como `float64`, lo cual es coherente ya que pueden involucrar decimales y operaciones aritméticas.

**CANTIDAD**  
Se mantiene como `float64`, lo cual es indispensable dado que existen cantidades fraccionadas (venta por peso).




## **2.3 Ingeniería de variables de negocio (venta neta, flags, precio unitario)**

In [4]:
# Asegurar columnas clave (por si alguien re-abrió y no guardó algunas)
if {"VENTA", "DESCUENTO"}.issubset(df.columns) and "VENTA_NETA" not in df.columns:
    df["VENTA_NETA"] = df["VENTA"] - df["DESCUENTO"]

# Promo flag
if "PROMO_FLAG" not in df.columns:
    if "OFERTA_ID_NUM" in df.columns:
        df["PROMO_FLAG"] = (df["OFERTA_ID_NUM"] > 0).astype(int)
    elif "OFERTA_ID" in df.columns:
        df["OFERTA_ID_NUM"] = pd.to_numeric(df["OFERTA_ID"], errors="coerce").fillna(0).astype(int)
        df["PROMO_FLAG"] = (df["OFERTA_ID_NUM"] > 0).astype(int)
    else:
        df["PROMO_FLAG"] = (df["DESCUENTO"].fillna(0) > 0).astype(int)

# Precio unitario neto (exploratorio)
if "PRECIO_UNITARIO_NETO" not in df.columns and {"VENTA_NETA", "CANTIDAD"}.issubset(df.columns):
    df["PRECIO_UNITARIO_NETO"] = np.where(df["CANTIDAD"] > 0, df["VENTA_NETA"] / df["CANTIDAD"], np.nan)

# Porcentaje de descuento (si podemos estimarlo)
# Nota: si VENTA es neta ya, esto cambia. Aquí asumimos VENTA bruto.
if {"VENTA", "DESCUENTO"}.issubset(df.columns):
    df["DESCUENTO_PCT"] = np.where(df["VENTA"] > 0, df["DESCUENTO"] / df["VENTA"], 0.0)

df[["VENTA", "DESCUENTO", "VENTA_NETA", "CANTIDAD", "PROMO_FLAG", "PRECIO_UNITARIO_NETO", "DESCUENTO_PCT"]].head(10)

,VENTA,DESCUENTO,VENTA_NETA,CANTIDAD,PROMO_FLAG,PRECIO_UNITARIO_NETO,DESCUENTO_PCT
0,36900,0.000,"36,900.000",2.061,0,"17,903.930",0.000
1,47900,0.000,"47,900.000",7.095,0,"6,751.233",0.000
2,30450,0.000,"30,450.000",0.984,0,"30,945.122",0.000
3,17750,0.000,"17,750.000",0.771,0,"23,022.049",0.000
4,12150,0.000,"12,150.000",0.402,0,"30,223.881",0.000
5,1200,0.000,"1,200.000",0.321,0,"3,738.318",0.000
6,17980,0.000,"17,980.000",0.792,0,"22,702.020",0.000
7,11850,0.000,"11,850.000",0.633,0,"18,720.379",0.000
8,31400,0.000,"31,400.000",5.402,0,"5,812.662",0.000
9,31800,0.000,"31,800.000",0.580,0,"54,827.586",0.000



## Consolidación de variables derivadas y creación de indicador porcentual de descuento

La salida confirma la consolidación de variables derivadas clave para análisis comercial y modelado, además de incorporar un indicador adicional que estandariza el efecto del descuento entre productos con precios distintos.

### Variables consolidadas en la tabla

La vista incluye:

- `VENTA`: valor bruto de la línea
- `DESCUENTO`: descuento aplicado en moneda
- `VENTA_NETA`: venta después de descuento
- `CANTIDAD`: unidades o cantidad fraccionada vendida
- `PROMO_FLAG`: indicador binario de promoción
- `PRECIO_UNITARIO_NETO`: aproximación de precio neto por unidad
- `DESCUENTO_PCT`: descuento relativo respecto a la venta bruta

### DESCUENTO_PCT: estandarización del impacto promocional

Se calcula como:

- `DESCUENTO_PCT = DESCUENTO / VENTA` (cuando `VENTA > 0`)

Este indicador es relevante porque permite comparar promociones de forma homogénea, incluso cuando se trata de productos con distintos niveles de precio. A nivel analítico, habilita:

- medición de intensidad promocional por producto o categoría
- segmentación por rangos de descuento (bajo, medio, alto)
- evaluación de relación entre descuento relativo y volumen vendido

En la muestra mostrada, `DESCUENTO_PCT = 0.000` en los primeros registros, consistente con `DESCUENTO = 0` y `PROMO_FLAG = 0`, lo cual confirma coherencia interna entre las variables derivadas.



## **2.4 Features de calendario (día semana, mes, fin de mes, quincena)**

In [5]:
# Features de calendario (derivadas)
df["DOW"] = df["FECHA"].dt.dayofweek               # 0=Lun ... 6=Dom
df["DOM"] = df["FECHA"].dt.day                      # día del mes
df["MES"] = df["FECHA"].dt.month
df["ANIO"] = df["FECHA"].dt.year
df["SEMANA_ANIO"] = df["FECHA"].dt.isocalendar().week.astype(int)

# Fin de mes
df["FIN_MES"] = df["FECHA"].dt.is_month_end.astype(int)

# Quincena (1: días 1-15, 2: 16-fin)
df["QUINCENA"] = np.where(df["DOM"] <= 15, 1, 2)

# Fin de semana
df["FIN_SEMANA"] = (df["DOW"] >= 5).astype(int)

df[["FECHA", "DOW", "DOM", "MES", "ANIO", "SEMANA_ANIO", "FIN_MES", "QUINCENA", "FIN_SEMANA"]].head(10)

,FECHA,DOW,DOM,MES,ANIO,SEMANA_ANIO,FIN_MES,QUINCENA,FIN_SEMANA
0,2026-01-01,3,1,1,2026,1,0,1,0
1,2026-01-01,3,1,1,2026,1,0,1,0
2,2026-01-01,3,1,1,2026,1,0,1,0
3,2026-01-01,3,1,1,2026,1,0,1,0
4,2026-01-01,3,1,1,2026,1,0,1,0
5,2026-01-01,3,1,1,2026,1,0,1,0
6,2026-01-01,3,1,1,2026,1,0,1,0
7,2026-01-01,3,1,1,2026,1,0,1,0
8,2026-01-01,3,1,1,2026,1,0,1,0
9,2026-01-01,3,1,1,2026,1,0,1,0



## Generación de variables de calendario para análisis temporal

La salida muestra la creación de variables temporales derivadas a partir de `FECHA`, con el objetivo de capturar patrones recurrentes en el comportamiento de ventas (estacionalidad, efectos de fin de mes y hábitos de consumo por día de semana).

### Variables creadas y su utilidad analítica

- `DOW` (day of week): codifica el día de la semana (0=Lunes, 6=Domingo).  
  Permite medir diferencias de ventas entre días hábiles y fines de semana, y capturar patrones semanales.

- `DOM` (día del mes): identifica el número de día dentro del mes.  
  Es útil para analizar ciclos intra-mensuales y relacionarlos con eventos de pago o abastecimiento.

- `MES` y `ANIO`: facilitan agregaciones mensuales/anuales y habilitan análisis de estacionalidad a mayor escala.

- `SEMANA_ANIO`: semana ISO del año.  
  Útil para modelado de series de tiempo con periodicidad semanal y para comparar semanas equivalentes entre años.

- `FIN_MES`: indicador binario de cierre de mes.  
  Captura efectos típicos de consumo asociados al final de mes, donde pueden ocurrir cambios en demanda o estrategias comerciales.

- `QUINCENA`: indicador de primera o segunda mitad del mes.  
  Es una variable relevante en contextos de nómina y pagos quincenales, permitiendo analizar si existe concentración de consumo en determinados periodos del mes.

- `FIN_SEMANA`: indicador binario para sábado o domingo.  
  Permite capturar incrementos o cambios de comportamiento asociados a compras de fin de semana.

### Lectura de la muestra observada

En los registros mostrados se evidencia coherencia temporal:

- `FECHA = 2026-01-01`
- `DOM = 1` (primer día del mes)
- `MES = 1`, `ANIO = 2026`
- `QUINCENA = 1` (primera mitad del mes)
- `FIN_MES = 0` (no es cierre de mes)
- `FIN_SEMANA = 0` (no corresponde a sábado o domingo)



## **2.5 Construcción de tablas agregadas para forecasting (día–tienda–producto y día–tienda–categoría)**

In [6]:
# Objetivo: construir series temporales agregadas a niveles útiles

# Helpers: columnas disponibles
has_store = "CENTRO" in df.columns
has_prod = "PLU_SAP" in df.columns
has_cat = "GRUPO_CATEG" in df.columns

# Agregación día–tienda–producto
group_cols_ptp = [c for c in ["FECHA", "CENTRO", "PLU_SAP"] if c in df.columns]
agg_ptp = (
    df.groupby(group_cols_ptp, as_index=False)
      .agg(
          VENTA_BRUTA=("VENTA", "sum"),
          DESCUENTO=("DESCUENTO", "sum"),
          VENTA_NETA=("VENTA_NETA", "sum"),
          UNIDADES=("CANTIDAD", "sum"),
          PROMO_LINES=("PROMO_FLAG", "sum"),
          LINES=("PROMO_FLAG", "size"),
          PRECIO_UNIT_MED=("PRECIO_UNITARIO_NETO", "median"),
          DESC_PCT_MED=("DESCUENTO_PCT", "median"),
      )
)

# Promo rate por agregación (proporción de líneas con promo)
agg_ptp["PROMO_RATE"] = np.where(agg_ptp["LINES"] > 0, agg_ptp["PROMO_LINES"] / agg_ptp["LINES"], 0.0)

agg_ptp.head(5), agg_ptp.shape

(       FECHA  CENTRO  PLU_SAP  VENTA_BRUTA  DESCUENTO  VENTA_NETA  UNIDADES  PROMO_LINES  LINES  PRECIO_UNIT_MED  DESC_PCT_MED  PROMO_RATE
 0 2026-01-01       1        3        36900      0.000  36,900.000     2.061            0      1       17,903.930         0.000       0.000
 1 2026-01-01       1        5       143700      0.000 143,700.000    21.285            0      3        6,751.233         0.000       0.000
 2 2026-01-01       1        7        60900      0.000  60,900.000     1.968            0      2       30,945.122         0.000       0.000
 3 2026-01-01       1        9        17750      0.000  17,750.000     0.771            0      1       23,022.049         0.000       0.000
 4 2026-01-01       1       11        24300      0.000  24,300.000     0.804            0      2       30,223.881         0.000       0.000,
 (228, 12))


## Construcción de serie agregada a nivel día–tienda–producto

El bloque presentado consolida la información transaccional al nivel operativo más útil para análisis de demanda: **producto por tienda por día**.

### Nivel de agregación construido

Columnas clave utilizadas:
- `FECHA`
- `CENTRO`
- `PLU_SAP`

Este nivel representa la granularidad adecuada para:

- Modelos de predicción por producto
- Análisis de rotación por tienda
- Evaluación de impacto promocional por SKU

### Métricas agregadas generadas

Para cada combinación día–tienda–producto se calculan:

- `VENTA_BRUTA`: suma de ventas antes de descuento
- `DESCUENTO`: suma total de descuentos
- `VENTA_NETA`: ingreso real consolidado
- `UNIDADES`: volumen total vendido
- `PROMO_LINES`: número de líneas con promoción
- `LINES`: número total de líneas (transacciones del SKU ese día)
- `PRECIO_UNIT_MED`: mediana del precio unitario neto
- `DESC_PCT_MED`: mediana del porcentaje de descuento

Adicionalmente:

- `PROMO_RATE`: proporción de líneas en promoción sobre el total de líneas

Esta métrica permite medir intensidad promocional de manera relativa y comparable entre productos.




## **2.6 Features temporales: lags y rolling windows (ventas y unidades)**

In [7]:
# IMPORTANTE: para lags/rolling necesitas múltiples fechas. Con la muestra de 1 día, esto queda en NaN.
# Aun así dejamos el pipeline listo para cuando llegue el dataset grande.

def add_time_features(df_ts: pd.DataFrame, group_keys: list, target_col: str, lags=(1,7,14), rolls=(7,14,28)) -> pd.DataFrame:
    out = df_ts.sort_values(group_keys + ["FECHA"]).copy()
    for L in lags:
        out[f"{target_col}_LAG_{L}"] = out.groupby(group_keys)[target_col].shift(L)
    for W in rolls:
        out[f"{target_col}_ROLLMEAN_{W}"] = out.groupby(group_keys)[target_col].shift(1).rolling(W).mean()
        out[f"{target_col}_ROLLSTD_{W}"] = out.groupby(group_keys)[target_col].shift(1).rolling(W).std()
    return out

# Ejemplo: features sobre VENTA_NETA y UNIDADES
if len(agg_ptp["FECHA"].unique()) > 1:
    agg_ptp_feat = add_time_features(agg_ptp, group_keys=[c for c in ["CENTRO", "PLU_SAP"] if c in agg_ptp.columns], target_col="VENTA_NETA")
    agg_ptp_feat = add_time_features(agg_ptp_feat, group_keys=[c for c in ["CENTRO", "PLU_SAP"] if c in agg_ptp.columns], target_col="UNIDADES")
else:
    agg_ptp_feat = agg_ptp.copy()
    print(" Solo hay 1 fecha en la muestra: lags/rolling quedarán vacíos hasta tener más histórico.")

agg_ptp_feat.head(5)

 Solo hay 1 fecha en la muestra: lags/rolling quedarán vacíos hasta tener más histórico.


,FECHA,CENTRO,PLU_SAP,VENTA_BRUTA,DESCUENTO,VENTA_NETA,UNIDADES,PROMO_LINES,LINES,PRECIO_UNIT_MED,DESC_PCT_MED,PROMO_RATE
0,2026-01-01,1,3,36900,0.000,"36,900.000",2.061,0,1,"17,903.930",0.000,0.000
1,2026-01-01,1,5,143700,0.000,"143,700.000",21.285,0,3,"6,751.233",0.000,0.000
2,2026-01-01,1,7,60900,0.000,"60,900.000",1.968,0,2,"30,945.122",0.000,0.000
3,2026-01-01,1,9,17750,0.000,"17,750.000",0.771,0,1,"23,022.049",0.000,0.000
4,2026-01-01,1,11,24300,0.000,"24,300.000",0.804,0,2,"30,223.881",0.000,0.000



## Preparación de features temporales (lags y rolling) para pronóstico

Se define una función (`add_time_features`) que, para cada combinación de agrupación (por ejemplo, `CENTRO` y `PLU_SAP`), construye:

- Rezagos (lags) del objetivo: `VENTA_NETA_LAG_1`, `LAG_7`, `LAG_14`
- Ventanas móviles (rolling) sobre el pasado (usando `shift(1)` para evitar fuga de información):
  - Promedio móvil: `ROLLMEAN_7`, `ROLLMEAN_14`, `ROLLMEAN_28`
  - Desviación estándar móvil: `ROLLSTD_7`, `ROLLSTD_14`, `ROLLSTD_28`

Este diseño es relevante porque captura:
- memoria de corto y mediano plazo
- patrones semanales (7 días) y quincenales (14 días)
- volatilidad de la demanda (rolling std), útil para inventarios y abastecimiento

### Mensaje de advertencia observado

Esto no representa un error del pipeline; es una validación correcta. Con un único día no existe información previa para calcular rezagos o promedios móviles, por lo que dichas columnas quedarían sin valores hasta disponer de múltiples fechas.

### Valor estratégico del enfoque

Aunque la muestra actual es de un solo día, este bloque deja lista la arquitectura para escalar cuando se cuente con el histórico completo. En la práctica, al incorporar más fechas, el mismo flujo producirá automáticamente:

- features temporales por producto y tienda
- señales de tendencia y estacionalidad
- indicadores de variabilidad y comportamiento reciente


## **2.7 Features de promociones: intensidad y efecto (descuento %, promo rate)**

In [7]:
# Intensidad de promo (monto y %)
# Nota: si tuvieras precio regular, podrías calcular elasticidades mejor. Aquí hacemos proxies.

agg_ptp_feat["DESCUENTO_PCT_APROX"] = np.where(agg_ptp_feat["VENTA_BRUTA"] > 0, agg_ptp_feat["DESCUENTO"] / agg_ptp_feat["VENTA_BRUTA"], 0.0)

# Señal de promo en agregación: si hubo al menos una línea en promo
agg_ptp_feat["PROMO_FLAG_AGG"] = (agg_ptp_feat["PROMO_LINES"] > 0).astype(int)

# Precio medio neto aproximado (valor neto / unidades)
agg_ptp_feat["PRECIO_MEDIO_NETO_APROX"] = np.where(agg_ptp_feat["UNIDADES"] > 0, agg_ptp_feat["VENTA_NETA"] / agg_ptp_feat["UNIDADES"], np.nan)

agg_ptp_feat[["FECHA", "CENTRO", "PLU_SAP", "VENTA_NETA", "UNIDADES", "PROMO_RATE", "DESCUENTO_PCT_APROX", "PROMO_FLAG_AGG", "PRECIO_MEDIO_NETO_APROX"]].head(10)

,FECHA,CENTRO,PLU_SAP,VENTA_NETA,UNIDADES,PROMO_RATE,DESCUENTO_PCT_APROX,PROMO_FLAG_AGG,PRECIO_MEDIO_NETO_APROX
0,2026-01-01,1,3,"36,900.000",2.061,0.000,0.000,0,"17,903.930"
1,2026-01-01,1,5,"143,700.000",21.285,0.000,0.000,0,"6,751.233"
2,2026-01-01,1,7,"60,900.000",1.968,0.000,0.000,0,"30,945.122"
3,2026-01-01,1,9,"17,750.000",0.771,0.000,0.000,0,"23,022.049"
4,2026-01-01,1,11,"24,300.000",0.804,0.000,0.000,0,"30,223.881"
5,2026-01-01,1,13,"7,200.000",1.926,0.000,0.000,0,"3,738.318"
6,2026-01-01,1,15,"17,980.000",0.792,0.000,0.000,0,"22,702.020"
7,2026-01-01,1,17,"23,700.000",1.266,0.000,0.000,0,"18,720.379"
8,2026-01-01,1,19,"31,400.000",5.402,0.000,0.000,0,"5,812.662"
9,2026-01-01,1,21,"31,800.000",0.580,0.000,0.000,0,"54,827.586"



## Construcción de indicadores de intensidad promocional y precio agregado

El bloque presentado fortalece el nivel agregado día–tienda–producto incorporando métricas que permiten medir de forma más estructurada el impacto comercial de las promociones y el comportamiento de precios.

### 1. DESCUENTO_PCT_APROX

Se calcula como:

DESCUENTO / VENTA_BRUTA

Este indicador permite medir la intensidad relativa del descuento a nivel agregado del SKU en el día. A diferencia del descuento absoluto, esta métrica es comparable entre productos de distintos rangos de precio.

En la muestra observada:
- DESCUENTO_PCT_APROX = 0.000
- Coherente con PROMO_RATE = 0.000

Esto confirma consistencia interna en el extracto analizado.

### 2. PROMO_FLAG_AGG

Se define como:

1 si hubo al menos una línea en promoción  
0 en caso contrario

Esta variable es útil para:

- Modelos donde interesa capturar simplemente presencia/ausencia de promoción.
- Análisis binario del efecto promocional sobre volumen.
- Simplificación de variables cuando no se requiere intensidad continua.

En la muestra, todos los valores son 0, lo cual es consistente con la ausencia de promociones en el día analizado.

### 3. PRECIO_MEDIO_NETO_APROX

Se calcula como:

VENTA_NETA / UNIDADES

Representa el precio promedio efectivo por unidad a nivel agregado del SKU en el día.

Este indicador es clave porque:

- Captura señal económica directa.
- Permite estudiar relación precio–volumen.
- Puede utilizarse como variable explicativa en modelos de demanda.

En la tabla se observan precios heterogéneos entre productos, lo cual es esperable dado que se trata de múltiples referencias.


## **2.8 Validaciones posteriores a agregación (consistencia y gaps)**

In [8]:
post_checks = {}

# 1) Venta neta negativa (raro sin devoluciones)
post_checks["venta_neta_negativa_agregada"] = int((agg_ptp_feat["VENTA_NETA"] < 0).sum())

# 2) Descuento > venta bruta agregada (raro)
post_checks["descuento_mayor_que_venta_bruta_agregada"] = int((agg_ptp_feat["DESCUENTO"] > agg_ptp_feat["VENTA_BRUTA"]).sum())

# 3) Unidades <= 0 (raro)
post_checks["unidades_le_0_agregadas"] = int((agg_ptp_feat["UNIDADES"] <= 0).sum())

# 4) Distribución de promo rate
post_checks["promo_rate_min"] = float(np.nanmin(agg_ptp_feat["PROMO_RATE"].values))
post_checks["promo_rate_max"] = float(np.nanmax(agg_ptp_feat["PROMO_RATE"].values))

post_checks

{'venta_neta_negativa_agregada': 0,
 'descuento_mayor_que_venta_bruta_agregada': 0,
 'unidades_le_0_agregadas': 0,
 'promo_rate_min': 0.0,
 'promo_rate_max': 1.0}


## Controles de consistencia posteriores a la agregación

Una vez construida la tabla agregada a nivel día–tienda–producto, se ejecutaron validaciones adicionales para garantizar que la transformación no introdujera inconsistencias numéricas o lógicas.

### Resultados de validación

- `venta_neta_negativa_agregada: 0`
- `descuento_mayor_que_venta_bruta_agregada: 0`
- `unidades_le_0_agregadas: 0`
- `promo_rate_min: 0.0`
- `promo_rate_max: 1.0`

### Interpretación ejecutiva

**1. Venta neta negativa = 0**  
No existen registros agregados con ingresos negativos. Esto confirma que la consolidación mantiene coherencia financiera.

**2. Descuento mayor que venta bruta = 0**  
Ningún SKU presenta descuento total superior a la venta bruta agregada. Esto descarta distorsiones contables tras la agregación.

**3. Unidades menores o iguales a cero = 0**  
Todos los productos presentan volumen positivo. No hay agregaciones con errores de suma ni inconsistencias operativas.

**4. PROMO_RATE dentro de rango válido (0.0 – 1.0)**  
La proporción de líneas en promoción se mantiene en el rango lógico esperado.  
Un máximo de 1.0 implica que, en algunos casos (en histórico futuro), un SKU podría venderse 100% bajo promoción en un día determinado, lo cual es matemáticamente consistente.



## **2.9 Dataset final de Fase 1 (tablas listas para EDA y modelado baseline)**

In [9]:
# Guardar tablas agregadas (listas para EDA y baseline)
out_ptp = DATA_PROCESSED / "ts_ventas_dia_tienda_producto.csv"
agg_ptp_feat.to_csv(out_ptp, index=False, encoding="utf-8")
print("Guardado:", out_ptp)

# (Opcional) también crear día–tienda–categoría si existe GRUPO_CATEG
if "GRUPO_CATEG" in df.columns:
    group_cols_ptc = [c for c in ["FECHA", "CENTRO", "GRUPO_CATEG"] if c in df.columns]
    agg_ptc = (
        df.groupby(group_cols_ptc, as_index=False)
          .agg(
              VENTA_BRUTA=("VENTA", "sum"),
              DESCUENTO=("DESCUENTO", "sum"),
              VENTA_NETA=("VENTA_NETA", "sum"),
              UNIDADES=("CANTIDAD", "sum"),
              PROMO_LINES=("PROMO_FLAG", "sum"),
              LINES=("PROMO_FLAG", "size"),
          )
    )
    agg_ptc["PROMO_RATE"] = np.where(agg_ptc["LINES"] > 0, agg_ptc["PROMO_LINES"] / agg_ptc["LINES"], 0.0)

    out_ptc = DATA_PROCESSED / "ts_ventas_dia_tienda_categoria.csv"
    agg_ptc.to_csv(out_ptc, index=False, encoding="utf-8")
    print("Guardado:", out_ptc)
else:
    print("No existe GRUPO_CATEG en el dataset; omitimos tabla día–tienda–categoría.")

Guardado: C:\Users\juana\olimpica_book\data\processed\ts_ventas_dia_tienda_producto.csv
Guardado: C:\Users\juana\olimpica_book\data\processed\ts_ventas_dia_tienda_categoria.csv



## Persistencia de series temporales listas para análisis y modelado

El bloque ejecutado formaliza uno de los entregables más importantes del pipeline: la generación y almacenamiento de tablas agregadas optimizadas para análisis exploratorio y modelos predictivos.

### Archivos generados

Se guardaron correctamente los siguientes archivos en `data/processed`:

- `ts_ventas_dia_tienda_producto.csv`
- `ts_ventas_dia_tienda_categoria.csv`

Esto implica que el proyecto ya cuenta con dos niveles estratégicos de agregación:

1. Día–Tienda–Producto  
   Nivel ideal para modelos de forecast por SKU y análisis detallado de elasticidad precio–demanda.

2. Día–Tienda–Categoría  
   Nivel más agregado, útil para:
   - Planeación comercial
   - Presupuestación
   - Evaluación de desempeño por línea de negocio
   - Modelos cuando el histórico por SKU sea insuficiente

### Valor estratégico de esta persistencia

Guardar estas tablas en la carpeta `processed` garantiza:

- Separación clara entre datos crudos y datos listos para analítica.
- Reproducibilidad del flujo.
- Facilidad para conectar estas tablas a dashboards o modelos sin reprocesar todo el pipeline.
- Escalabilidad futura cuando se incorporen más fechas y centros.

